In [27]:
import csv
import pandas as pd
titanic = pd.DataFrame(pd.read_csv('/Users/yogeshsah/Desktop/ML project/titanic/train.csv'))
print(titanic.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None


In [28]:
# Data Cleaning
titanic_clean = titanic.dropna(subset=["Embarked"])
#titanic_clean = titanic_clean.drop(["Name","Ticket"],axis=1)
count_og = titanic["Cabin"].isna().sum()
titanic_clean["Cabin"] = (titanic_clean["Cabin"].isna()!= True)
count = (titanic_clean["Cabin"] == False).sum()
print(count_og, count)
titanic_train = titanic_clean["Survived"]
print(titanic_clean.info())

687 687
<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    int64  
 1   Survived     889 non-null    int64  
 2   Pclass       889 non-null    int64  
 3   Name         889 non-null    object 
 4   Sex          889 non-null    object 
 5   Age          712 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Ticket       889 non-null    object 
 9   Fare         889 non-null    float64
 10  Cabin        889 non-null    bool   
 11  Embarked     889 non-null    object 
dtypes: bool(1), float64(2), int64(5), object(4)
memory usage: 84.2+ KB
None


/var/folders/51/kyqw99j15rb_t15v9ntyb8cc0000gn/T/ipykernel_78862/2232209434.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  titanic_clean["Cabin"] = (titanic_clean["Cabin"].isna()!= True)


In [29]:
# checking correlation for numerical columns
corr_matrix = titanic_clean[["Age","SibSp","Parch","Fare","Survived"]].corr()["Survived"]
print(corr_matrix.sort_values(ascending = True))
# for categorical columns
cat_cols = ["Pclass","Sex","Embarked","Cabin"]
for col in cat_cols:
    results = titanic_clean.groupby(col)["Survived"].mean()
    print(results)

Age        -0.082446
SibSp      -0.034040
Parch       0.083151
Fare        0.255290
Survived    1.000000
Name: Survived, dtype: float64
Pclass
1    0.626168
2    0.472826
3    0.242363
Name: Survived, dtype: float64
Sex
female    0.740385
male      0.188908
Name: Survived, dtype: float64
Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64
Cabin
False    0.299854
True     0.663366
Name: Survived, dtype: float64


In [30]:
#feature engineering
titanic_clean["Age_bins"] = pd.cut(titanic_clean["Age"],bins=[0,18,35,65,100],labels=[1, 2, 3, 4]).astype(float)
titanic_clean["fam_size"] = titanic_clean["SibSp"]+titanic_clean["Parch"]+1
corr_matrix = titanic_clean[["Age_bins","fam_size","Survived"]].corr()["Survived"]
print(corr_matrix)
titanic_clean["Title"] = titanic_clean["Name"].str.extract(r'([A-Za-z)]+)\.')
print(titanic_clean["Title"].value_counts())

Age_bins   -0.087471
fam_size    0.018277
Survived    1.000000
Name: Survived, dtype: float64
Title
Mr          517
Miss        181
Mrs         124
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64


/var/folders/51/kyqw99j15rb_t15v9ntyb8cc0000gn/T/ipykernel_78862/3361245326.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  titanic_clean["Age_bins"] = pd.cut(titanic_clean["Age"],bins=[0,18,35,65,100],labels=[1, 2, 3, 4]).astype(float)
/var/folders/51/kyqw99j15rb_t15v9ntyb8cc0000gn/T/ipykernel_78862/3361245326.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  titanic_clean["fam_size"] = titanic_clean["SibSp"]+titanic_clean["Parch"]+1
/var/folders/51/kyqw99j15rb_t15v9ntyb8cc0000gn/T/ipykernel_78862/33

In [31]:
# preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
encoder = OneHotEncoder()
sex_encoder = OneHotEncoder()
#sex_encoded = sex_encoder.fit_transform(titanic[["Sex"]])
##print(sex_encoder.categories_)

port_encoder = OneHotEncoder()
#port_encoded = port_encoder.fit_transform(titanic_clean[["Embarked"]])
#print(port_encoder.categories_)

class_encoder = OrdinalEncoder()
#class_encoded = class_encoder.fit_transform(titanic_clean[["Pclass"]])
#print(class_encoder.categories_)


imputer = SimpleImputer(strategy="median")




In [32]:
#pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
   ("std_scaler", StandardScaler())
])

In [33]:
#prepare the data
from sklearn.compose import ColumnTransformer
num_col=["Age","SibSp","Parch","Fare"]
cat_col=["Sex","Embarked"]
titanic_clean=titanic_clean[num_col+cat_col+["Pclass"]]
full_pipeline = ColumnTransformer([
    ("num",num_pipeline,num_col),
    ("encoder",OneHotEncoder(),cat_col),
    ("class_encoder",OrdinalEncoder(),["Pclass"])
])
titanic_pr = full_pipeline.fit_transform(titanic_clean)
feature_names = full_pipeline.get_feature_names_out()
titanic_pr = pd.DataFrame(titanic_pr, columns=feature_names)
print(titanic_pr.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 889 entries, 0 to 888
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num__Age               889 non-null    float64
 1   num__SibSp             889 non-null    float64
 2   num__Parch             889 non-null    float64
 3   num__Fare              889 non-null    float64
 4   encoder__Sex_female    889 non-null    float64
 5   encoder__Sex_male      889 non-null    float64
 6   encoder__Embarked_C    889 non-null    float64
 7   encoder__Embarked_Q    889 non-null    float64
 8   encoder__Embarked_S    889 non-null    float64
 9   class_encoder__Pclass  889 non-null    float64
dtypes: float64(10)
memory usage: 69.6 KB
None


In [34]:
# selecting a model
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
sgd_clf = SGDClassifier(random_state=42)
y_predictions = cross_val_predict(sgd_clf,titanic_pr,titanic_train,cv=3)
f1=f1_score(titanic_train,y_predictions)
roc_auc = roc_auc_score(titanic_train,y_predictions)
print("F1:",f1)
print("ROC:",roc_auc)

F1: 0.6699346405228758
ROC: 0.7404505518054216


In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
forest_clf = RandomForestClassifier(random_state=42)
y_predictions = cross_val_predict(forest_clf,titanic_pr,titanic_train,cv=3)
roc_auc = roc_auc_score(titanic_train,y_predictions)
print("F1:",f1)
print("ROC:",roc_auc)

F1: 0.6699346405228758
ROC: 0.783582449373192


In [36]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score
models ={
    "SGD" : SGDClassifier(random_state=42),
    "Gradient Boost": GradientBoostingClassifier(random_state=42),
    "Random Forest":  RandomForestClassifier(random_state=42),
    "SVM":            SVC(probability=True, random_state=42),
}
for name,model in models.items():
    scores=cross_val_score(model,titanic_pr,titanic_train,cv=5,scoring="roc_auc")
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")
for name,model in models.items():
    scores=cross_val_score(model,titanic_pr,titanic_train,cv=5,scoring="accuracy")
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")
for name,model in models.items():
    scores=cross_val_score(model,titanic_pr,titanic_train,cv=5,scoring="f1")
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

SGD: 0.821 (+/- 0.017)
Gradient Boost: 0.868 (+/- 0.037)
Random Forest: 0.853 (+/- 0.037)
SVM: 0.848 (+/- 0.023)
SGD: 0.755 (+/- 0.011)
Gradient Boost: 0.830 (+/- 0.022)
Random Forest: 0.811 (+/- 0.023)
SVM: 0.827 (+/- 0.015)
SGD: 0.658 (+/- 0.049)
Gradient Boost: 0.762 (+/- 0.038)
Random Forest: 0.745 (+/- 0.039)
SVM: 0.759 (+/- 0.025)
